# 1. 월별 물동량 장기 성장/계절성/환적 구조 파악
## 의의: 부산항만공사에서 제공하는 '부산항만공사_물동량 예측.csv' 데이터를 활용하고, 분석하는 과정을 수행해본다.

## 1. (장기) 성장 추세
부산항이 여러 해에 걸쳐 지속적으로 성장하고 있는가?
- 단, 특정 한 해의 물동량이 작년보다 증가했다고 해서 '지속적으로' 성장했다고 판단하기는 이름. 여러가지 변수에 의해 특정 한 해만 증가했을 가능성이 있기 때문.
- (장기) 성장 추세를 파악하여 부산항의 성장이 지속, 정체, 감소하고 있는지 파악 가능.

In [27]:
import pandas as pd
import numpy as np

In [28]:
CSV_PATH = '../prData1.csv'
df = pd.read_csv(CSV_PATH, encoding='cp949')

In [29]:
df.head()

,연도,월,연월,수출입구분,수출입구분설명,물동량
0,2003,2,2003-02,OT,수출환적,"161,074.50"
1,2003,11,2003-11,IT,수입환적,"170,733.50"
2,2006,1,2006-01,OT,수출환적,"203,348.25"
3,2018,1,2018-01,OO,수출,"413,797.75"
4,2013,6,2013-06,OO,수출,"389,650.25"


In [30]:
df.shape

(1376, 6)

In [31]:
df.dtypes

연도           int64
월            int64
연월             str
수출입구분          str
수출입구분설명        str
물동량        float64
dtype: object

In [32]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1376 entries, 0 to 1375
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   연도       1376 non-null   int64  
 1   월        1376 non-null   int64  
 2   연월       1376 non-null   str    
 3   수출입구분    1376 non-null   str    
 4   수출입구분설명  1376 non-null   str    
 5   물동량      1376 non-null   float64
dtypes: float64(1), int64(2), str(3)
memory usage: 64.6 KB


In [33]:
df.columns.to_list()

['연도', '월', '연월', '수출입구분', '수출입구분설명', '물동량']

In [34]:
df.isna().sum() # 결측치 없음

연도         0
월          0
연월         0
수출입구분      0
수출입구분설명    0
물동량        0
dtype: int64

In [35]:
df.describe()

,연도,월,물동량
count,"1,376.00","1,376.00","1,376.00"
mean,"2,009.98",6.47,"324,784.23"
std,8.86,3.46,"119,286.83"
min,"1,992.00",1.00,"90,300.00"
25%,"2,003.00",3.00,"222,387.25"
50%,"2,010.00",6.00,"322,195.50"
75%,"2,017.25",9.00,"420,758.44"
max,"2,025.00",12.00,"635,437.00"


In [36]:
df['연월'] = pd.to_datetime(
    df['연월'],
    format='%Y-%m',
    errors='coerce'
)

df.dtypes

연도                  int64
월                   int64
연월         datetime64[us]
수출입구분                 str
수출입구분설명               str
물동량               float64
dtype: object

In [37]:
origin = df.copy()
# df(원본) - str이 us로 수정된 상태의 원본을 origin(보관용)으로 복사
# 앞으로 df로 작업 진행

In [38]:
df['연도'].unique()

array([2003, 2006, 2018, 2013, 2012, 2017, 2014, 2015, 2019, 2022, 1993,
       1995, 2001, 2004, 2007, 2010, 2024, 1994, 1996, 1997, 1998, 1999,
       2002, 2005, 2008, 2025, 2021, 2009, 2011, 2016, 1992, 2020, 2023,
       2000])

In [61]:
df['수출입구분설명'].unique()

<StringArray>
['수출', '수입', '수입환적', '수출환적']
Length: 4, dtype: str

In [39]:
df = df.sort_values(
    by=['연도', '월', '연월']
)
df

,연도,월,연월,수출입구분,수출입구분설명,물동량
602,1992,1,1992-01-01,OO,수출,"104,567.00"
864,1992,1,1992-01-01,II,수입,"92,005.00"
491,1992,2,1992-02-01,II,수입,"91,669.00"
735,1992,2,1992-02-01,OO,수출,"103,903.00"
282,1992,3,1992-03-01,OO,수출,"137,082.00"
...,...,...,...,...,...,...
1232,2025,1,2025-01-01,II,수입,"468,006.50"
50,2025,2,2025-02-01,OO,수출,"437,906.75"
207,2025,2,2025-02-01,OT,수출환적,"599,626.75"
858,2025,2,2025-02-01,II,수입,"392,441.25"


In [40]:
df.head(20)

,연도,월,연월,수출입구분,수출입구분설명,물동량
602,1992,1,1992-01-01,OO,수출,"104,567.00"
864,1992,1,1992-01-01,II,수입,"92,005.00"
491,1992,2,1992-02-01,II,수입,"91,669.00"
735,1992,2,1992-02-01,OO,수출,"103,903.00"
282,1992,3,1992-03-01,OO,수출,"137,082.00"
641,1992,3,1992-03-01,II,수입,"105,276.00"
432,1992,4,1992-04-01,OO,수출,"130,353.00"
603,1992,4,1992-04-01,II,수입,"99,261.00"
214,1992,5,1992-05-01,II,수입,"95,773.00"
1163,1992,5,1992-05-01,OO,수출,"126,586.00"


In [41]:
df_detach = df[
    ['연월', '수출입구분', '수출입구분설명', '물동량']
].copy()
df_detach

,연월,수출입구분,수출입구분설명,물동량
602,1992-01-01,OO,수출,"104,567.00"
864,1992-01-01,II,수입,"92,005.00"
491,1992-02-01,II,수입,"91,669.00"
735,1992-02-01,OO,수출,"103,903.00"
282,1992-03-01,OO,수출,"137,082.00"
...,...,...,...,...
1232,2025-01-01,II,수입,"468,006.50"
50,2025-02-01,OO,수출,"437,906.75"
207,2025-02-01,OT,수출환적,"599,626.75"
858,2025-02-01,II,수입,"392,441.25"


In [42]:
# 흩어져 있는 각 연월의 물동량을 취합
monthly = (
    df_detach
    .groupby('연월', as_index=False)['물동량']
    .sum()
)
monthly.head(20)

,연월,물동량
0,1992-01-01,"196,572.00"
1,1992-02-01,"195,572.00"
2,1992-03-01,"242,358.00"
3,1992-04-01,"229,614.00"
4,1992-05-01,"222,359.00"
5,1992-06-01,"219,336.00"
6,1992-07-01,"225,557.00"
7,1992-08-01,"213,642.00"
8,1992-09-01,"213,136.00"
9,1992-10-01,"221,729.00"


In [43]:
monthly.shape

(398, 2)

In [44]:
monthly['전월대비증감률'] = (
    monthly['물동량']
    .pct_change() * 100
)

monthly.head(20)

,연월,물동량,전월대비증감률
0,1992-01-01,"196,572.00",NaN
1,1992-02-01,"195,572.00",-0.51
2,1992-03-01,"242,358.00",23.92
3,1992-04-01,"229,614.00",-5.26
4,1992-05-01,"222,359.00",-3.16
5,1992-06-01,"219,336.00",-1.36
6,1992-07-01,"225,557.00",2.84
7,1992-08-01,"213,642.00",-5.28
8,1992-09-01,"213,136.00",-0.24
9,1992-10-01,"221,729.00",4.03


In [45]:
# +/- 를 넘나드는 증감률을 통해 전월대비증감률 만으로는 성장추세를 알기 어렵다는 걸 확인
# 때문에 12개월평균을 추가 확인
# 12개월치 데이터가 모이기 전은 NaN으로 표시
monthly['12개월평균'] = (
    monthly['물동량']
    .rolling(12)
    .mean()
)

monthly.head(20)

,연월,물동량,전월대비증감률,12개월평균
0,1992-01-01,"196,572.00",NaN,NaN
1,1992-02-01,"195,572.00",-0.51,NaN
2,1992-03-01,"242,358.00",23.92,NaN
3,1992-04-01,"229,614.00",-5.26,NaN
4,1992-05-01,"222,359.00",-3.16,NaN
5,1992-06-01,"219,336.00",-1.36,NaN
6,1992-07-01,"225,557.00",2.84,NaN
7,1992-08-01,"213,642.00",-5.28,NaN
8,1992-09-01,"213,136.00",-0.24,NaN
9,1992-10-01,"221,729.00",4.03,NaN


In [46]:
monthly['전월대비증감률'] = (
    monthly['전월대비증감률']
    .round(2) # 소수점 2자리까지만
)
monthly.head(20)

,연월,물동량,전월대비증감률,12개월평균
0,1992-01-01,"196,572.00",NaN,NaN
1,1992-02-01,"195,572.00",-0.51,NaN
2,1992-03-01,"242,358.00",23.92,NaN
3,1992-04-01,"229,614.00",-5.26,NaN
4,1992-05-01,"222,359.00",-3.16,NaN
5,1992-06-01,"219,336.00",-1.36,NaN
6,1992-07-01,"225,557.00",2.84,NaN
7,1992-08-01,"213,642.00",-5.28,NaN
8,1992-09-01,"213,136.00",-0.24,NaN
9,1992-10-01,"221,729.00",4.03,NaN


In [47]:
yearly_trend = monthly[
    monthly['연월'].dt.month == 12
][['연월', '12개월평균']]

yearly_trend

,연월,12개월평균
11,1992-12-01,"219,614.50"
23,1993-12-01,"245,102.92"
35,1994-12-01,"298,543.08"
47,1995-12-01,"340,861.15"
59,1996-12-01,"363,502.92"
71,1997-12-01,"400,930.12"
83,1998-12-01,"442,609.29"
95,1999-12-01,"471,740.17"
107,2000-12-01,"531,859.81"
119,2001-12-01,"672,733.48"


In [48]:
pd.options.display.float_format = '{:,.2f}'.format

yearly_trend

,연월,12개월평균
11,1992-12-01,"219,614.50"
23,1993-12-01,"245,102.92"
35,1994-12-01,"298,543.08"
47,1995-12-01,"340,861.15"
59,1996-12-01,"363,502.92"
71,1997-12-01,"400,930.12"
83,1998-12-01,"442,609.29"
95,1999-12-01,"471,740.17"
107,2000-12-01,"531,859.81"
119,2001-12-01,"672,733.48"


In [49]:
monthly.head(50)

,연월,물동량,전월대비증감률,12개월평균
0,1992-01-01,"196,572.00",NaN,NaN
1,1992-02-01,"195,572.00",-0.51,NaN
2,1992-03-01,"242,358.00",23.92,NaN
3,1992-04-01,"229,614.00",-5.26,NaN
4,1992-05-01,"222,359.00",-3.16,NaN
5,1992-06-01,"219,336.00",-1.36,NaN
6,1992-07-01,"225,557.00",2.84,NaN
7,1992-08-01,"213,642.00",-5.28,NaN
8,1992-09-01,"213,136.00",-0.24,NaN
9,1992-10-01,"221,729.00",4.03,NaN


In [50]:
monthly.tail(20)

,연월,물동량,전월대비증감률,12개월평균
378,2023-07-01,"1,896,403.75",-0.30,"1,858,313.65"
379,2023-08-01,"1,893,300.75",-0.16,"1,858,434.52"
380,2023-09-01,"1,965,505.50",3.81,"1,892,226.92"
381,2023-10-01,"1,895,968.25",-3.54,"1,897,777.77"
382,2023-11-01,"1,993,664.75",5.15,"1,914,794.58"
383,2023-12-01,"1,916,583.00",-3.87,"1,929,459.00"
384,2024-01-01,"1,992,396.50",3.96,"1,942,216.50"
385,2024-02-01,"1,879,271.50",-5.68,"1,952,266.92"
386,2024-03-01,"2,143,065.75",14.04,"1,957,728.48"
387,2024-04-01,"2,046,147.00",-4.52,"1,959,534.02"


## 2. 계절성 = 성수기,비수기
특정한 시기마다 일정한 증가나 감소가 반복적으로 나타나는 현상을 의미.(진짜 계절적 의미의 분기가 아님.)
- 물동량이 많이 발생하는 시기를 예상할 수 있다면 전체 운영계획을 미리 세우기에 용이.

In [53]:
season = monthly.copy()

In [54]:
season['월'] = season['연월'].dt.month

season.head(15)

,연월,물동량,전월대비증감률,12개월평균,월
0,1992-01-01,"196,572.00",NaN,NaN,1
1,1992-02-01,"195,572.00",-0.51,NaN,2
2,1992-03-01,"242,358.00",23.92,NaN,3
3,1992-04-01,"229,614.00",-5.26,NaN,4
4,1992-05-01,"222,359.00",-3.16,NaN,5
5,1992-06-01,"219,336.00",-1.36,NaN,6
6,1992-07-01,"225,557.00",2.84,NaN,7
7,1992-08-01,"213,642.00",-5.28,NaN,8
8,1992-09-01,"213,136.00",-0.24,NaN,9
9,1992-10-01,"221,729.00",4.03,NaN,10


In [ ]:
season['평균대비지수'] = (
    season['물동량'] / season['12개월평균'] * 100
)

season['평균대비지수'] = (
    season['평균대비지수'].round(2)
)

season.head(20)
# 각각의 연월이 각 해의 12개월 치 평균보다 얼마나 높거나 낮았는지 확인
# 아직 계절지수라고 확정 짓기는 어려움.
# 최종 계절지수는 ex) '3월이 지속적으로 높거나 낮았는가?'

,연월,물동량,전월대비증감률,12개월평균,월,평균대비지수
0,1992-01-01,"196,572.00",NaN,NaN,1,NaN
1,1992-02-01,"195,572.00",-0.51,NaN,2,NaN
2,1992-03-01,"242,358.00",23.92,NaN,3,NaN
3,1992-04-01,"229,614.00",-5.26,NaN,4,NaN
4,1992-05-01,"222,359.00",-3.16,NaN,5,NaN
5,1992-06-01,"219,336.00",-1.36,NaN,6,NaN
6,1992-07-01,"225,557.00",2.84,NaN,7,NaN
7,1992-08-01,"213,642.00",-5.28,NaN,8,NaN
8,1992-09-01,"213,136.00",-0.24,NaN,9,NaN
9,1992-10-01,"221,729.00",4.03,NaN,10,NaN


In [56]:
season_index = (
    season
    .groupby('월', as_index=False)['평균대비지수']
    .mean()
)

season_index

,월,평균대비지수
0,1,100.99
1,2,93.73
2,3,109.18
3,4,106.74
4,5,107.71
5,6,103.79
6,7,105.64
7,8,101.75
8,9,99.87
9,10,103.75


In [57]:
season_index['평균대비지수'] = (
    season_index['평균대비지수'].round(2)
)

In [58]:
season_index = season_index.rename(
    columns={'평균대비지수': '계절지수'}
)

In [59]:
season_index

,월,계절지수
0,1,100.99
1,2,93.73
2,3,109.18
3,4,106.74
4,5,107.71
5,6,103.79
6,7,105.64
7,8,101.75
8,9,99.87
9,10,103.75


In [ ]:
season_index['평균대비차이'] = (
    season_index['계절지수'] - 100
).round(2)

season_index # 좀 더 시작적 직관성을 위해 '평균대비차이' 추가

,월,계절지수,평균대비차이
0,1,100.99,0.99
1,2,93.73,-6.27
2,3,109.18,9.18
3,4,106.74,6.74
4,5,107.71,7.71
5,6,103.79,3.79
6,7,105.64,5.64
7,8,101.75,1.75
8,9,99.87,-0.13
9,10,103.75,3.75


`"100을 기준으로 계절지수가 낮게 나온 달은 2월, 9월이다."`
- 해석 예시
1. 2월은 달이 짧아 물동량이 적다.
2. 2월과 9월에는 설날과 추석 연휴로 항만이 쉬었을 가능성이 있다.

## 3. 환적 구조 파악
전체 컨테이너 물량만 보기 보다 항목 별로 구분해서 각각의 물동량을 보는 것이 포인트.
- 전체 물동량은 증가한 것으로 보일 수 있으나 어떤 물동량이 늘어났는지에 따라 가지는 의미가 다르기 때문.

In [67]:
df_detach['수출입구분설명'].unique()

<StringArray>
['수출', '수입', '수입환적', '수출환적']
Length: 4, dtype: str

In [68]:
trans = df_detach.copy()

trans['구분'] = trans['수출입구분설명'].replace({
    '수입': '수출입',
    '수출': '수출입',
    '수입환적': '환적',
    '수출환적': '환적'
})

In [69]:
trans[
    ['수출입구분설명', '구분']
].drop_duplicates()

,수출입구분설명,구분
602,수출,수출입
864,수입,수출입
372,수입환적,환적
1356,수출환적,환적


In [70]:
trans_monthly = (
    trans
    .groupby(['연월', '구분'], as_index=False)['물동량']
    .sum()
)

trans_monthly.head(10)

,연월,구분,물동량
0,1992-01-01,수출입,"196,572.00"
1,1992-02-01,수출입,"195,572.00"
2,1992-03-01,수출입,"242,358.00"
3,1992-04-01,수출입,"229,614.00"
4,1992-05-01,수출입,"222,359.00"
5,1992-06-01,수출입,"219,336.00"
6,1992-07-01,수출입,"225,557.00"
7,1992-08-01,수출입,"213,642.00"
8,1992-09-01,수출입,"213,136.00"
9,1992-10-01,수출입,"221,729.00"


In [ ]:
# 환적이 언제부터 등장하는지 확인
trans_monthly[
    trans_monthly['구분'] == '환적'
].head(10)

,연월,구분,물동량
109,2001-01-01,환적,"241,026.25"
111,2001-02-01,환적,"187,758.75"
113,2001-03-01,환적,"257,623.00"
115,2001-04-01,환적,"243,019.25"
117,2001-05-01,환적,"228,487.00"
119,2001-06-01,환적,"234,847.75"
121,2001-07-01,환적,"246,543.25"
123,2001-08-01,환적,"258,836.25"
125,2001-09-01,환적,"269,432.75"
127,2001-10-01,환적,"257,844.00"


In [ ]:
# 각각 몇행씩 존재하는지 확인
trans['수출입구분설명'].value_counts()

수출입구분설명
수출      398
수입      398
수입환적    290
수출환적    290
Name: count, dtype: int64

In [74]:
# 환적이 2001-01 부터 등장하므로 해당 시점부터 계산
trans_2001 = trans_monthly[
    trans_monthly['연월'].dt.year >= 2001
].copy()

trans_2001.head(10)

,연월,구분,물동량
108,2001-01-01,수출입,"388,518.75"
109,2001-01-01,환적,"241,026.25"
110,2001-02-01,수출입,"390,455.00"
111,2001-02-01,환적,"187,758.75"
112,2001-03-01,수출입,"458,648.25"
113,2001-03-01,환적,"257,623.00"
114,2001-04-01,수출입,"438,011.25"
115,2001-04-01,환적,"243,019.25"
116,2001-05-01,수출입,"440,076.75"
117,2001-05-01,환적,"228,487.00"


In [75]:
# 최종적으로 환적의 성장 기여도까지 확인할 거라면?
# 월별 계산보다 연도 별로 묶는 방향으로 접근
trans_2001['연도'] = trans_2001['연월'].dt.year

trans_yearly = (
    trans_2001
    .groupby(['연도', '구분'], as_index=False)['물동량']
    .sum()
)

trans_yearly.head(10)

,연도,구분,물동량
0,2001,수출입,"5,129,694.00"
1,2001,환적,"2,943,107.75"
2,2002,수출입,"5,566,104.00"
3,2002,환적,"3,887,984.00"
4,2003,수출입,"6,157,128.25"
5,2003,환적,"4,251,414.25"
6,2004,수출입,"6,698,925.00"
7,2004,환적,"4,791,840.00"
8,2005,수출입,"6,663,670.75"
9,2005,환적,"5,179,724.00"


In [76]:
# pivot 세로로 쌓인 항목을 가로로 펼치는 기능
trans_table = trans_yearly.pivot(
    index='연도',
    columns='구분',
    values='물동량'
)

trans_table.head()

구분,수출입,환적
연도,,
2001,"5,129,694.00","2,943,107.75"
2002,"5,566,104.00","3,887,984.00"
2003,"6,157,128.25","4,251,414.25"
2004,"6,698,925.00","4,791,840.00"
2005,"6,663,670.75","5,179,724.00"


In [ ]:
# 전체물동량 계산
trans_table['전체물동량'] = (
    trans_table['수출입'] + trans_table['환적']
)

trans_table

구분,수출입,환적,전체물동량
연도,,,
2001,"5,129,694.00","2,943,107.75","8,072,801.75"
2002,"5,566,104.00","3,887,984.00","9,454,088.00"
2003,"6,157,128.25","4,251,414.25","10,408,542.50"
2004,"6,698,925.00","4,791,840.00","11,490,765.00"
2005,"6,663,670.75","5,179,724.00","11,843,394.75"
2006,"6,830,813.25","5,208,295.25","12,039,108.50"
2007,"7,450,248.75","5,811,591.00","13,261,839.75"
2008,"7,644,942.00","5,808,188.00","13,453,130.00"
2009,"6,607,973.75","5,372,636.25","11,980,610.00"


## 2001년 데이터로 모의 계산
- 전체 8,072,801.75가 100%짜리 한 덩어리
그중에서 환적이 2,943,107.75만큼 포함.

수출입       5,129,694.00
환적         2,943,107.75
──────────────────────
전체         8,072,801.75

- 
환적          2,943,107.75
──────  =  ────────────────  = 0.3646
전체          8,072,801.75

`2001년 전체 물동량 100% 가운데 환적이 36.46%를 차지함`
0.3646 × 100 = 36.46%

In [87]:
# 환적비율 = 환적물동량 ÷ 전체물동량 × 100
# 알고 싶은 대상 ÷ 기준이 되는 전체

trans_table['환적비율'] = (
    trans_table['환적']
    / trans_table['전체물동량']
    * 100
).round(2)

trans_table.head(25)

구분,수출입,환적,전체물동량,환적비율
연도,,,,
2001,"5,129,694.00","2,943,107.75","8,072,801.75",36.46
2002,"5,566,104.00","3,887,984.00","9,454,088.00",41.12
2003,"6,157,128.25","4,251,414.25","10,408,542.50",40.85
2004,"6,698,925.00","4,791,840.00","11,490,765.00",41.70
2005,"6,663,670.75","5,179,724.00","11,843,394.75",43.74
2006,"6,830,813.25","5,208,295.25","12,039,108.50",43.26
2007,"7,450,248.75","5,811,591.00","13,261,839.75",43.82
2008,"7,644,942.00","5,808,188.00","13,453,130.00",43.17
2009,"6,607,973.75","5,372,636.25","11,980,610.00",44.84


In [ ]:
# 2025년은 2개월 밖에 없는데 공신력 있는 값인가?
df_detach[
    df_detach['연월'].dt.year == 2025
]['연월'].unique()

<DatetimeArray>
['2025-01-01 00:00:00', '2025-02-01 00:00:00']
Length: 2, dtype: datetime64[us]

In [89]:
growth = trans_table.copy()

In [ ]:
# trans_table을 CSV로 저장해두고, 두 번째 파일에서는 그 저장된 표만 불러오는 방법.

trans_table.to_csv(
    'trans_table.csv',
    encoding='cp949'
)